# 12 - Improved DeepLog-style LSTM

A rare-event rule and smaller k for HDFS, then the same model on BGL windows.

In [1]:
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import evaluate
from src.models.lstm import DeepLogLSTM, build_vocab, flag, make_windows, next_event_ranks, train
from src.models.unsupervised import time_split

DOCS, PROCESSED = ROOT / "docs", ROOT / "data" / "processed"
H, SEED = 10, 42
MODEL_CONFIG = {"embed_dim": 32, "hidden_size": 64, "num_layers": 2}


def evenly_spaced(items, n):
    return [items[i] for i in np.linspace(0, len(items) - 1, n).round().astype(int)]


def known_events(blocks, m, by="occurrences"):
    """Events seen at least m times (or in at least m blocks) in training; rarer ones count as unseen."""
    counts = Counter(e for s in blocks for e in (set(s) if by == "blocks" else s))
    return {e for e, n in counts.items() if n >= m}


def fit(blocks, vocab):
    """Same settings as notebook 08: 10% of blocks for validation, Adam, early stopping."""
    val_mask = np.random.default_rng(SEED).random(len(blocks)) < 0.1
    X_tr, y_tr, _ = make_windows([s for s, v in zip(blocks, val_mask) if not v], vocab, H)
    X_val, y_val, _ = make_windows([s for s, v in zip(blocks, val_mask) if v], vocab, H)
    torch.manual_seed(SEED)
    model = DeepLogLSTM(num_events=len(vocab), **MODEL_CONFIG)
    _, seconds = train(model, X_tr, y_tr, X_val, y_val, epochs=30, patience=3, seed=SEED)
    print(f"Training windows: {len(y_tr):,}   training time: {seconds:.1f} s   "
          f"parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


def metrics_row(y_true, pred, **extra):
    m = evaluate(y_true, pred)
    return {**extra, "precision": m["precision"], "recall": m["recall"], "f1": m["f1"], "false_alarm_rate": m["false_alarm_rate"]}

## Part A: HDFS

The spread 20,000-block model from notebook 08. The model is run once on the test set; each (m, k) only changes the rule applied to its predictions.

In [2]:
hdfs = pd.read_parquet(PROCESSED / "hdfs_sequences.parquet")
train_part, test = time_split(hdfs, train_frac=0.7)
normal_train = train_part[train_part["Label"] == 0]["EventSequence"].tolist()
blocks = evenly_spaced(normal_train, 20000)
test_seqs, y_test = test["EventSequence"].tolist(), test["Label"].to_numpy()

saved = torch.load(ROOT / "models" / "hdfs_lstm_spread20k.pt")
vocab = saved["vocab"]
model = DeepLogLSTM(num_events=len(vocab), **MODEL_CONFIG)
model.load_state_dict(saved["state_dict"])
assert set(saved["train_events"]) == known_events(blocks, 1)  # same training set as notebook 08

start = time.perf_counter()
ranks, seq_idx = next_event_ranks(model, test_seqs, vocab, H)
print(f"Model pass over {len(ranks):,} test windows: {time.perf_counter() - start:.1f} s")

Model pass over 2,686,487 test windows: 24.0 s


In [3]:
event_counts = Counter(e for s in blocks for e in s)
block_counts = Counter(e for s in blocks for e in set(s))
print(f"E33 appears {event_counts['E33']} times, in {block_counts['E33']} different blocks, in the spread 20,000-block training set")
print(f"Rarest events in training: {', '.join(f'{e} ({n})' for e, n in sorted(event_counts.items(), key=lambda x: x[1])[:4])}\n")

rows = []
for m in [1, 5, 10]:
    known = known_events(blocks, m)
    for k in [1, 2, 3, 5, 9]:
        rows.append(metrics_row(y_test, flag(test_seqs, ranks, seq_idx, known, k), rule="occurrences", m=m, k=k))

hdfs_results = pd.DataFrame(rows)
best_m = int(hdfs_results.loc[hdfs_results["f1"].idxmax(), "m"])
print(f"Best F1 with m = {best_m}")
hdfs_results.style.format({c: "{:.3f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]}).hide(axis="index")

E33 appears 12 times, in 12 different blocks, in the spread 20,000-block training set
Rarest events in training: E8 (10), E9 (10), E33 (12), E11 (114)



Best F1 with m = 1


rule,m,k,precision,recall,f1,false_alarm_rate
occurrences,1,1,0.030,0.980,0.059,0.693
occurrences,1,2,0.249,0.721,0.371,0.048
occurrences,1,3,0.431,0.604,0.503,0.018
occurrences,1,5,0.889,0.506,0.645,0.001
occurrences,1,9,0.953,0.400,0.563,0.000
occurrences,5,1,0.030,0.980,0.059,0.693
occurrences,5,2,0.249,0.721,0.371,0.048
occurrences,5,3,0.431,0.604,0.503,0.018
occurrences,5,5,0.889,0.506,0.645,0.001
occurrences,5,9,0.953,0.400,0.563,0.000


m has no effect here: every event in the training set occurs at least 10 times (the rarest, E8 and E9, exactly 10, and E33 12), so m = 5 and m = 10 mark no extra events as unseen and give the same results as m = 1.

### Rare-event rule by number of blocks

An event found in fewer than m different training blocks counts as unseen, instead of fewer than m occurrences.

In [4]:
print(f"Rarest events by blocks: {', '.join(f'{e} ({n})' for e, n in sorted(block_counts.items(), key=lambda x: x[1])[:4])}\n")

rows = []
for m in [2, 5, 10]:
    known = known_events(blocks, m, by="blocks")
    print(f"m = {m:2d} blocks: events now counted as unseen {sorted(set(block_counts) - known) or 'none'}")
    for k in [3, 5, 9]:
        rows.append(metrics_row(y_test, flag(test_seqs, ranks, seq_idx, known, k), rule="blocks", m=m, k=k))

block_results = pd.DataFrame(rows)
pd.concat([hdfs_results, block_results], ignore_index=True).round(4).to_csv(DOCS / "hdfs_lstm_improved.csv", index=False)
block_results.style.format({c: "{:.3f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]}).hide(axis="index")

Rarest events by blocks: E9 (3), E8 (3), E33 (12), E11 (72)

m =  2 blocks: events now counted as unseen none


m =  5 blocks: events now counted as unseen ['E8', 'E9']


m = 10 blocks: events now counted as unseen ['E8', 'E9']


rule,m,k,precision,recall,f1,false_alarm_rate
blocks,2,3,0.431,0.604,0.503,0.018
blocks,2,5,0.889,0.506,0.645,0.001
blocks,2,9,0.953,0.400,0.563,0.000
blocks,5,3,0.431,0.604,0.503,0.018
blocks,5,5,0.889,0.506,0.645,0.001
blocks,5,9,0.953,0.400,0.563,0.000
blocks,10,3,0.431,0.604,0.503,0.018
blocks,10,5,0.889,0.506,0.645,0.001
blocks,10,9,0.953,0.400,0.563,0.000


Counting blocks changes nothing either: E33 is in 12 blocks, so it is still counted as seen for m ≤ 10, and the only events that become unseen (E8 and E9, 3 blocks each) appear in 24 test blocks that the LSTM already flags.

## Part B: BGL

Same 70/30 split by window index as notebooks 10 and 11. The model trains on 20,000 normal windows spread evenly over the training part, with h = 10 inside each 20-event window.

In [5]:
bgl = pd.read_parquet(PROCESSED / "bgl_sequences.parquet")
train_part_b, test_b = time_split(bgl, train_frac=0.7, order_col="window")
windows = evenly_spaced(train_part_b[train_part_b["Label"] == 0]["EventSequence"].tolist(), 20000)
vocab_b = build_vocab(pd.read_csv(PROCESSED / "bgl_templates.csv")["EventId"])
print(f"Event types: {len(vocab_b)} in total, {len(known_events(windows, 1))} in the 20,000 training windows")

model_b = fit(windows, vocab_b)

Event types: 391 in total, 186 in the 20,000 training windows


epoch  1  train loss 0.2194  val loss 0.2591


epoch  2  train loss 0.1591  val loss 0.1895


epoch  3  train loss 0.1300  val loss 0.1572


epoch  4  train loss 0.1096  val loss 0.1328


epoch  5  train loss 0.1014  val loss 0.1231


... (12 outputs left out)

epoch 18  train loss 0.0795  val loss 0.1072


epoch 19  train loss 0.0792  val loss 0.1083


epoch 20  train loss 0.0783  val loss 0.1083


epoch 21  train loss 0.0779  val loss 0.1080
early stopping after epoch 21
Training windows: 342,988   training time: 336.8 s   parameters: 96,392


In [6]:
test_seqs_b, y_test_b = test_b["EventSequence"].tolist(), test_b["Label"].to_numpy()

start = time.perf_counter()
ranks_b, seq_idx_b = next_event_ranks(model_b, test_seqs_b, vocab_b, H)
model_seconds = time.perf_counter() - start
print(f"Model pass over {len(ranks_b):,} test windows: {model_seconds:.1f} s (shared by all k)")

known_b = known_events(windows, best_m)
preds_b, rows = {}, []
for k in [1, 3, 5, 9]:
    start = time.perf_counter()
    preds_b[k] = flag(test_seqs_b, ranks_b, seq_idx_b, known_b, k)
    seconds = model_seconds + time.perf_counter() - start
    rows.append(metrics_row(y_test_b, preds_b[k], m=best_m, k=k, detection_seconds=seconds))

bgl_results = pd.DataFrame(rows)
bgl_results.round(4).to_csv(DOCS / "bgl_lstm_results.csv", index=False)
bgl_results.style.format({c: "{:.3f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]} | {"detection_seconds": "{:.1f}"}).hide(axis="index")

Model pass over 1,353,180 test windows: 17.0 s (shared by all k)


m,k,detection_seconds,precision,recall,f1,false_alarm_rate
1,1,17.1,0.142,0.992,0.248,0.484
1,3,17.1,0.157,0.992,0.270,0.431
1,5,17.1,0.166,0.992,0.284,0.403
1,9,17.1,0.172,0.992,0.294,0.384


Recall on anomalous test windows with an event type that never appears in the training part (normal or anomalous), and on the other anomalous windows. Same split as notebook 11.

In [7]:
seen_in_train = {e for s in train_part_b["EventSequence"] for e in s}
has_unseen = np.array([any(e not in seen_in_train for e in s) for s in test_seqs_b])
anomalous = y_test_b == 1
print(f"Anomalous test windows with an unseen event type: {(anomalous & has_unseen).sum():,} of {anomalous.sum():,}")

pd.DataFrame({
    f"k={k}": {"recall, windows with unseen event type": pred[anomalous & has_unseen].mean(),
               "recall, other anomalous windows": pred[anomalous & ~has_unseen].mean()}
    for k, pred in preds_b.items()
}).round(3)

Anomalous test windows with an unseen event type: 2,539 of 5,312


,k=1,k=3,k=5,k=9
"recall, windows with unseen event type",1.000,1.000,1.000,1.000
"recall, other anomalous windows",0.986,0.986,0.986,0.986
